In [1]:
#SETUP
from datetime import datetime
from delta import DeltaTable
import ConnectionConfig as cc
cc.setupEnvironment()

In [2]:
#SETUP
spark = cc.startLocalCluster("dimCustomerIncrementalLoad")
spark.getActiveSession()

## Incremental load 
* Setting the parameters

In [3]:

run_timestamp = datetime.now()

In [4]:
cc.set_connectionProfile("VeloDB")

#### Read source table


In [5]:
#EXTRACT
df_velo_users =spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_subscriptions = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscriptions") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptionid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

df_subscription_types = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "subscription_types") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "subscriptiontypeid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

In [6]:
#TRANSFORM
df_velo_users.createOrReplaceTempView("operational_velo_users")
df_subscriptions.createOrReplaceTempView("operational_subscriptions")
df_subscription_types.createOrReplaceTempView("operational_subscription_types")

In [7]:
#TRANSFORM
df_dim_customers = spark.sql("""
    SELECT 
        monotonically_increasing_id() as source_customer_SK,
        u.userid as source_customer_id,
        u.name as source_customer,
        CONCAT(u.street, ' ', u.number, ', ', u.zipcode, ' ', u.city, ', ', u.country_code) as source_address,
        st.description as source_subscription_type,
        s.validfrom as source_validform,
        md5(concat(u.name, source_address)) as source_md5
    FROM operational_velo_users u
        JOIN operational_subscriptions s 
                ON u.userid = s.userid
        JOIN operational_subscription_types st 
                ON s.subscriptiontypeid = st.subscriptiontypeid
    WHERE s.validfrom = (
        SELECT MAX(validfrom) 
        FROM operational_subscriptions 
        WHERE userid = u.userid)
""")

df_dim_customers.createOrReplaceTempView("dimCustomer_new")

In [8]:
#TRANSFORM
dt_dimCustomer = DeltaTable.forPath(spark, "./spark-warehouse/dimCustomer")
dt_dimCustomer.toDF().createOrReplaceTempView("dimCustomer_current")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
spark.sql("""
   select * from dimCustomer_new
""").show()

+------------------+------------------+------------------+--------------------+------------------------+----------------+--------------------+
|source_customer_SK|source_customer_id|   source_customer|      source_address|source_subscription_type|source_validform|          source_md5|
+------------------+------------------+------------------+--------------------+------------------------+----------------+--------------------+
|                 0|                 2| van der Zee Julia|bbbhhhhhhhhhhhhh ...|                   MAAND|      2021-10-05|512575af1f4c99c9b...|
|                 1|                 1|       Bouman Lars|SchoolStraat 2030...|                     DAG|      2020-12-14|dfdda69c4d0ef02dd...|
|                 2|                 4|    Willems Angela|Graaf Joseph de P...|                     DAG|      2019-04-08|6dd6cbbdd0f1d4ad2...|
|                 3|                 5|  Heijnen Patricia|Meylstraat 111 , ...|                    JAAR|      2022-05-16|e4ac66703279e7945...|

##### 2 DETECT CHANGES


In [9]:
#TRANSFORM
detectedChanges = spark.sql("""
    select * 
    from dimCustomer_new source
        left outer join dimCustomer_current dwh 
            on dwh.customer_id = source.source_customer_id
            and dwh.current == true
    where dwh.customer_id is null or dwh.md5 <> source.source_md5
""")
detectedChanges.createOrReplaceTempView("detectedChanges")

#DEBUG CODE TO SHOW CONTENT OF DETECTED CHANGES
detectedChanges.show()

+------------------+------------------+---------------+--------------------+------------------------+----------------+--------------------+-----------+-----------+-----------+--------------------+---------+-----------------+-------------------+-------------------+----------+---------------+--------------------+-------+
|source_customer_SK|source_customer_id|source_customer|      source_address|source_subscription_type|source_validform|          source_md5|customer_SK|customer_id|       name|             address|     city|subscription_type|          scd_start|            scd_end| validform|subscription_id|                 md5|current|
+------------------+------------------+---------------+--------------------+------------------------+----------------+--------------------+-----------+-----------+-----------+--------------------+---------+-----------------+-------------------+-------------------+----------+---------------+--------------------+-------+
|                 1|                 

##### 3 TRANSOFRM TO UPSERTS


In [10]:
#TRANSFORM 
#zoekt de grootste waarde, met de max functie en die doe je +1 en dan zet je die als je nieuwe sk
max_customer_SK = spark.sql("SELECT COALESCE(MAX(customer_SK), 0) AS max_customer_SK FROM dimCustomer_current").collect()[0]['max_customer_SK']


df_upsert = spark.sql(f"""
    select  
       {max_customer_SK} as customer_SK,
        source_customer_id as customer_id,
        source_customer as name,
        source_address as address,
        source_subscription_type as subscription_type,
        to_timestamp('{run_timestamp}') as scd_start,
        to_timestamp('2600-01-01', 'yyyy-MM-dd') as scd_end,
        true as current,
        source_md5 as md5
    FROM detectedChanges
    union all
        select  
            customer_SK,
            customer_id,
            name,
            address,
            subscription_type,
            scd_start,
            to_timestamp('{run_timestamp}') as scd_end,
            false as current,
            md5
        FROM detectedChanges
        where current is not null
""")
df_upsert.createOrReplaceTempView("upsert")

In [11]:
#DEBUG CODE TO SHOW CONTENT OF UPSERT
spark.sql("""
    select * from upsert
""").show()

+-----------+-----------+-----------+--------------------+-----------------+--------------------+--------------------+-------+--------------------+
|customer_SK|customer_id|       name|             address|subscription_type|           scd_start|             scd_end|current|                 md5|
+-----------+-----------+-----------+--------------------+-----------------+--------------------+--------------------+-------+--------------------+
| 8589994849|          1|Bouman Lars|SchoolStraat 2030...|              DAG|2024-10-21 22:29:...| 2600-01-01 00:00:00|   true|dfdda69c4d0ef02dd...|
|          2|          1|Bouman Lars|tURNhout straat 1...|              DAG| 1999-01-01 00:00:00|2024-10-21 22:29:...|  false|14abf5c3c6b0ded67...|
+-----------+-----------+-----------+--------------------+-----------------+--------------------+--------------------+-------+--------------------+



In [12]:
#CREATE MERGE 
spark.sql("""
    MERGE INTO dimCustomer_current as target
    using upsert as source on target.customer_id = source.customer_id 
        and source.current = false 
        and target.current = true
    WHEN MATCHED THEN UPDATE SET scd_end = source.scd_end, current = source.current
    WHEN NOT MATCHED THEN INSERT (customer_SK, customer_id, name, address, subscription_type,  scd_start, scd_end, current, md5) values (source.customer_SK, source.customer_id, source.name, source.address, source.subscription_type, source.scd_start, source.scd_end, source.current, source.md5)
""")

#DEBUG CODE TO SHOW CONTENT OF DIMENSION
dt_dimCustomer.toDF().sort("customer_id", "scd_start").show(100)

+-----------+-----------+--------------------+--------------------+--------------------+-----------------+--------------------+--------------------+----------+---------------+--------------------+-------+
|customer_SK|customer_id|                name|             address|                city|subscription_type|           scd_start|             scd_end| validform|subscription_id|                 md5|current|
+-----------+-----------+--------------------+--------------------+--------------------+-----------------+--------------------+--------------------+----------+---------------+--------------------+-------+
|          2|          1|         Bouman Lars|tURNhout straat 1...|           Antwerpen|              DAG| 1999-01-01 00:00:00|2024-10-21 22:29:...|2020-12-14|              3|14abf5c3c6b0ded67...|  false|
| 8589994849|          1|         Bouman Lars|SchoolStraat 2030...|                NULL|              DAG|2024-10-21 22:29:...| 2600-01-01 00:00:00|      NULL|           NULL|dfdda

In [13]:
spark.stop()